# Hair & Scalp Care Advisor - training notebook v4 (leak-free, source-robust)

What is new compared with v3
1. **Leak-free split**: near-duplicate photos are grouped and never sit in both train and test (v3 had 62 of 71 test photos of the second dandruff set duplicated in training).
2. **Honest metric**: AUROC measured *inside one photo source* (`roboflow_hairdata`), so the model cannot win by recognising where a photo came from.
3. **Controlled experiments**, one change at a time: standard aug -> strong aug -> + balanced sources -> + source-blind training. The winner is chosen on out-of-fold (training-pool) predictions, **never on the test set**.
4. **Familiarity gate**, so the app can say "not sure" for photos unlike the training data.

**Before you run anything**: Settings -> Accelerator **GPU T4 x2** (or P100), **Internet On**. Attach two datasets (+ Add Input): `code` (zip containing `src/`, `scripts/`) and `images` (zip containing `balding/`, `dryness_dandruff/`, `healthy_scalp/`, `extra_scalp/`). Run cells top to bottom. Total time: roughly 2-3 hours.

## 1. Find the code and the images
This cell looks inside the attached datasets (and unpacks any `.zip` that Kaggle left unextracted).
If it fails, read the "Attached inputs" lines it prints: an empty list means you have not clicked **+ Add Input** yet.

In [ ]:
import os, glob, zipfile

def find_roots(base):
    code_root = raw_root = None
    for dirpath, dirnames, filenames in os.walk(base):
        if code_root is None and os.path.isfile(os.path.join(dirpath, "src", "train.py")):
            code_root = dirpath
        if raw_root is None and "balding" in dirnames and "dryness_dandruff" in dirnames:
            raw_root = dirpath
        if code_root and raw_root:
            break
    return code_root, raw_root

inputs = os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else []
print("Attached inputs:", inputs if inputs else "NONE - click '+ Add Input' on the right and attach both datasets")

CODE_ROOT, RAW_ROOT = find_roots("/kaggle/input")

if not (CODE_ROOT and RAW_ROOT):
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    print("Unextracted zip files found:", zips)
    for z in zips:
        out = os.path.join("/kaggle/working/unzipped", os.path.splitext(os.path.basename(z))[0])
        if not os.path.isdir(out):
            print("Unpacking", z, "...")
            with zipfile.ZipFile(z) as zf:
                zf.extractall(out)
    code2, raw2 = find_roots("/kaggle/working/unzipped")
    CODE_ROOT, RAW_ROOT = CODE_ROOT or code2, RAW_ROOT or raw2

print("CODE_ROOT:", CODE_ROOT)
print("RAW_ROOT :", RAW_ROOT)

if not (CODE_ROOT and RAW_ROOT):
    print("\nFolders visible under /kaggle/input (3 levels):")
    for dirpath, dirnames, filenames in os.walk("/kaggle/input"):
        depth = dirpath.count(os.sep) - 2
        if depth <= 3:
            print("  " * depth + os.path.basename(dirpath) + "/", f"({len(filenames)} files)")
    raise SystemExit("Stopped: code and/or images not found. See the listing above.")

for extra in ["healthy_scalp", "extra_scalp"]:
    print(f"{extra:<14}", "found" if os.path.isdir(os.path.join(RAW_ROOT, extra)) else "MISSING (scalp model quality will suffer without healthy_scalp)")

## 2. Copy the code to a writable place and run the smoke test (about 1 minute)
This uses fake images. It only proves the code runs; it says nothing about accuracy. It must print `SMOKE TEST PASSED`.

In [ ]:
import shutil
WORK = "/kaggle/working/code"
shutil.rmtree(WORK, ignore_errors=True)
for d in ["src", "scripts"]:
    shutil.copytree(os.path.join(CODE_ROOT, d), os.path.join(WORK, d))
!python {WORK}/scripts/smoke_test.py

## 3. Build the labels (leak-free) and PROVE there is no leakage

In [ ]:
!python {WORK}/scripts/build_labels.py --raw-root {RAW_ROOT} --out-dir /kaggle/working/data

In [ ]:
os.makedirs("/kaggle/working/results", exist_ok=True)
!python {WORK}/scripts/audit_data.py --raw-root {RAW_ROOT} --labels /kaggle/working/data/scalp_labels.csv --out /kaggle/working/results/audit_scalp.json | head -14
!python {WORK}/scripts/audit_data.py --raw-root {RAW_ROOT} --labels /kaggle/working/data/balding_labels.csv --out /kaggle/working/results/audit_balding.json | head -14
print("\nEXPECTED: near_duplicate_pairs_across_splits = 0 and test_images_with_a_near_duplicate_in_train_or_val = 0 in both.")

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image

scalp = pd.read_csv("/kaggle/working/data/scalp_labels.csv")
sources = list(scalp["source"].unique())
fig, axes = plt.subplots(len(sources), 6, figsize=(16, 2.7 * len(sources)), squeeze=False)
for r, src in enumerate(sources):
    sample = scalp[scalp["source"] == src].sample(min(6, (scalp["source"] == src).sum()), random_state=1)
    for c in range(6):
        ax = axes[r][c]; ax.axis("off")
        if c < len(sample):
            ax.imshow(Image.open(os.path.join(RAW_ROOT, sample.iloc[c]["filepath"])).convert("RGB"))
    axes[r][0].set_title(src, loc="left", fontsize=11)
plt.tight_layout(); plt.show()

## 4. Balding model: two recipes, pick by validation loss
Both train on the Kaggle train split; the **validation** split picks the winner. The test split is untouched until the final evaluation.

In [ ]:
for name, aug in [("B0_standard", "standard"), ("B1_strong", "strong")]:
    print("=" * 20, name)
    !python {WORK}/src/train.py --task balding --labels /kaggle/working/data/balding_labels.csv --img-dir {RAW_ROOT} --out-dir /kaggle/working/exp/{name} --aug {aug}

In [ ]:
import json
vl = {n: json.load(open(f"/kaggle/working/exp/{n}/balding_training_summary.json"))["folds"][0]["val_logloss"] for n in ["B0_standard", "B1_strong"]}
print("validation log-loss (lower is better):", vl)
BALDING_WINNER = min(vl, key=vl.get)
print("Balding winner:", BALDING_WINNER)

## 5. Scalp model: four controlled experiments (5-fold, grouped)
Each experiment changes ONE thing:
- **E0_baseline**: standard augmentation (what v3 did, but with the leak-free split)
- **E1_strong**: + strong style augmentation (grayscale, JPEG quality, resolution, noise, stronger colour)
- **E2_strong_balanced**: + each photo source contributes equally to every batch
- **E3_source_blind**: + a reversed-gradient "which source?" guesser so the features carry no source information

Look at the lines `style-free (inside roboflow_hairdata): AUROC=...` in each run: that is the number that matters.

In [ ]:
EXPS = {
  "E0_baseline":        "--aug standard",
  "E1_strong":          "--aug strong",
  "E2_strong_balanced": "--aug strong --balance-sources",
  "E3_source_blind":    "--aug strong --balance-sources --source-adv 1.0",
}
for name, flags in EXPS.items():
    print("=" * 25, name, flags)
    !python {WORK}/src/train.py --task scalp --labels /kaggle/working/data/scalp_labels.csv --img-dir {RAW_ROOT} --out-dir /kaggle/working/exp/{name} {flags}

In [ ]:
import json, numpy as np
rows = {}
for n in EXPS:
    s = json.load(open(f"/kaggle/working/exp/{n}/scalp_training_summary.json"))["style_free_oof"]
    per = {c: s[c].get("roboflow_hairdata", {}).get("auroc") for c in ["dryness", "dandruff", "low_density"]}
    vals = [v for v in per.values() if v is not None]
    rows[n] = (per, float(np.mean(vals)) if vals else float("nan"))
print(f"{'experiment':<22}{'dryness':>9}{'dandruff':>10}{'low_dens':>10}{'mean':>8}   (out-of-fold, inside roboflow_hairdata)")
for n, (per, m) in rows.items():
    print(f"{n:<22}{per['dryness']!s:>9}{per['dandruff']!s:>10}{per['low_density']!s:>10}{m:>8.3f}")
SCALP_WINNER = max(rows, key=lambda k: rows[k][1])
print("\nScalp winner (by out-of-fold style-free AUROC, NOT by test set):", SCALP_WINNER)

## 6. Final evaluation on the untouched test split
Only the baseline (for the before/after comparison) and the winners are evaluated. Read `style-free check` first.

In [ ]:
winners = {"balding": BALDING_WINNER, "scalp": SCALP_WINNER}
for exp in sorted({"E0_baseline", SCALP_WINNER}):
    !python {WORK}/src/evaluate.py --task scalp --labels /kaggle/working/data/scalp_labels.csv --img-dir {RAW_ROOT} --checkpoint /kaggle/working/exp/{exp}/scalp_model.pt --results-dir /kaggle/working/results/scalp_{exp}
!python {WORK}/src/evaluate.py --task balding --labels /kaggle/working/data/balding_labels.csv --img-dir {RAW_ROOT} --checkpoint /kaggle/working/exp/{BALDING_WINNER}/balding_model.pt --results-dir /kaggle/working/results/balding_{BALDING_WINNER}

## 7. Install the winners and add the familiarity gate

In [ ]:
import shutil
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
for task, exp in winners.items():
    for f in [f"{task}_model.pt", f"{task}_oof_predictions.csv", f"{task}_training_summary.json"]:
        shutil.copy(f"/kaggle/working/exp/{exp}/{f}", f"/kaggle/working/checkpoints/{f}")
!python {WORK}/scripts/build_gate.py --task scalp --labels /kaggle/working/data/scalp_labels.csv --img-dir {RAW_ROOT} --checkpoint /kaggle/working/checkpoints/scalp_model.pt
!python {WORK}/scripts/build_gate.py --task balding --labels /kaggle/working/data/balding_labels.csv --img-dir {RAW_ROOT} --checkpoint /kaggle/working/checkpoints/balding_model.pt

## 8. Summary files for the app and for your report
`accuracy_summary.json` feeds the app's "How accurate is this?" panel (verified numbers only).

In [ ]:
!python {WORK}/scripts/make_accuracy_summary.py --metrics /kaggle/working/results/scalp_{SCALP_WINNER}/metrics.json --source roboflow_hairdata --out /kaggle/working/accuracy_summary.json
!python {WORK}/scripts/make_accuracy_summary.py --metrics /kaggle/working/results/balding_{BALDING_WINNER}/metrics.json --source kaggle_balding --out /kaggle/working/accuracy_summary.json --append
!python {WORK}/scripts/compare_experiments.py --results-root /kaggle/working/results --source roboflow_hairdata


## 9. How to read the results honestly
- The **style-free AUROC** (inside `roboflow_hairdata`) is the number to quote. The pooled AUROC includes easy "healthy vs stock photo" comparisons and is optimistic.
- The hairdata test part is small (36 photos), so confidence intervals are wide: say so.
- If E1/E2/E3 are not clearly better than E0, report that honestly; "augmentation did not help" is a valid finding.
- None of this measures ordinary phone photos. State that as the main limitation.

## 10. Package everything to download

In [ ]:
for folder in ["checkpoints", "results", "data"]:
    os.makedirs(f"/kaggle/working/{folder}", exist_ok=True)
!cd /kaggle/working && zip -r -q hair_advisor_outputs_v4.zip checkpoints results data accuracy_summary.json exp/*/scalp_training_summary.json exp/*/balding_training_summary.json
print("Download hair_advisor_outputs_v4.zip from the Output tab.")